# Direct Preference Optimization (DPO) Fine-Tuning Tutorial 🚀

### Lesson Notes: DPO Kya Hai Aur Kaise Kaam Karta Hai?

**Direct Preference Optimization (DPO)** ek aisi technique hai jo model ko human preferences ke sath align karti hai bina complex Reinforcement Learning (jaise PPO) ke. 

**DPO ki workflow simple hai:**
1. **Prompt**: Jo question user poochta hai.
2. **Chosen Response**: Jo answer positive/behtar hai (jise model ko seekhna chahiye).
3. **Rejected Response**: Jo answer galat/unhelpful hai (jise model ko avoid karna chahiye).

Is notebook mein hum seekhenge ki kaise custom pharmacy dataset (`pharma_preference_data.csv`) ka use karke model ko DPO ke sath fine-tune kiya jata hai.

In [1]:
# Pehle TRL (Transformer Reinforcement Learning) aur bitsandbytes dependencies install karte hain
!uv pip install -U trl bitsandbytes --quiet

## 1. Setup & Imports (लाइब्रेरी और टोकनाइजर सेटअप) 🛠️

Sabse pehle hum training ke liye zaroori libraries import karenge aur base tokenizer select karenge.

In [2]:
# DPO training aur model integration ke liye PEFT aur transformers modules import karte hain
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import load_dataset
import torch

In [3]:
# Base model name define karte hain
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [4]:
# Tokenizer load karte hain aur verify karte hain ki padding settings correct hain
tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

## 2. Load Instruction-Tuned Model (पुराने मॉडल की जांच) 🔍

Pehle hum check karenge ki hamara pichla instruction-tuned model (`tinyllama-lora-instruction`) kya output de raha hai, taaki DPO training ke baad comparison kar sakein.

In [5]:
# Pichle chapter ke instruction-tuned checkpoint model path reference karte hain
model_path = "./tinyllama-lora-instruction"

In [6]:
# Instruction tuned checkpoint ko load karte hain directly bfloat16 accuracy settings par
instruction_model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.bfloat16, device_map="auto")

In [7]:
# Evaluation ke liye sample prompt define karte hain
prompt = "Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry."

In [8]:
# Input prompt ko tokenise karke Target device (MPS) par load karte hain
inputs = tokenizer(prompt, return_tensors="pt").to("mps")

In [9]:
# Model output generate karte hain. max_length config warning clear karne ke liye use None
instruction_model.generation_config.max_length = None
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [10]:
# Output decode karke print karte hain
print("\nInstruction-Tuned Model Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 3. Direct Preference Optimization Setup (डीपीओ की तैयारी) 🛠️

Ab hum DPO implementation start karenge. DPO training start karne ke liye hum Base Model load karenge aur instruction checkpoints ko usme merge karenge, phir use target benchmark as reference model banayenge.

In [11]:
# DPO trainer modules aur parameters import karte hain
from trl import DPOTrainer
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import PeftModel, LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import torch

In [12]:
# Base model name define karte hain
base_model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [13]:
# Instruction adapter path
instruction_checkpoint = "./tinyllama-lora-instruction/"

In [14]:
# CSV format preference data ko load karte hain
dataset = load_dataset("csv", data_files="./pharma_preference_data.csv")["train"]

In [15]:
# Base tokenizer load karte hain
tokenizer = AutoTokenizer.from_pretrained(base_model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [16]:
# LoRA adapter configuration define karte hain rank=8 ke sath
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [17]:
# Base model ko bfloat16 (precision type) format mein load karte hain directly
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

In [18]:
# Instruction checkpoints load karke base model par mount karte hain
model = PeftModel.from_pretrained(model, instruction_checkpoint)

In [19]:
# Adapters weights ko merge aur base structure ko unload karte hain. DPO starts from a merged base model.
model = model.merge_and_unload()

## 4. DPO Dataset Formatting (डेटा फ़ॉर्मेटिंग) 📂

DPO library ko processing ke liye specific schema keys chahiye: `prompt`, `chosen` aur `rejected`. Hum maps mapping ke through data format karenge.

In [20]:
# DPO requirements ke validation format mapping design
def process_data(example):
    return {
        "prompt": f"### Instruction:\n{example['prompt']}\n### Response:\n",
        "chosen": example["chosen"],
        "rejected": example["rejected"]
    }

dataset = dataset.map(process_data)
print("Sample DPO Formatted Prompt:", dataset[0]['prompt'])

## 5. DPO Hyperparameters & Configuration (डीपीओ सेटिंग्स) ⚙️

Latest TRL versions support DPOConfig. Hum use configuration feed karenge.

In [21]:
from trl import DPOConfig

# DPO specific hyper-parameters define karte hain. report_to='none' integration validation ValueError clear rakhega
dpo_args = DPOConfig(
    output_dir="./tinyllama-preference",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    bf16=True, # MPS numerical gradients safety
    logging_steps=1,
    report_to="none", # Resolves ValueError: None is not supported error
    beta=0.1, # KL divergence coefficient multiplier
    max_length=256,
    remove_unused_columns=False
)

## 6. DPO Trainer Setup and Launch (डीपीओ ट्रेनिंग लांच) 🚀

Hum custom base model (`model`) pass karenge aur trainer use automatic wrapped structure (`peft_config`) ke sath initialize kar dega.

In [22]:
# DPOTrainer initialization setup. We pass processing_class=tokenizer directly
trainer = DPOTrainer(
    model=model, # Merged base instruction model
    ref_model=None, # PEFT adapter disabled model is used for reference logic under the hood
    args=dpo_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=lora_config
)

# DPO training execute karte hain!
trainer.train()

## 7. Save and Test Aligned Model (डीपीओ रिस्पांस टेस्टिंग) 📈

Training complete hone ke baad saved model checkpoints reload karke comparison verify karte hain.

In [23]:
# Fine-tuned adapters save karte hain
trainer.save_model("./tinyllama-preference-final")
tokenizer.save_pretrained("./tinyllama-preference-final")

# Load model for direct evaluation
dpo_model = AutoModelForCausalLM.from_pretrained(
    "./tinyllama-preference-final",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

test_prompt = "### Instruction:\nExplain the mechanism of action of Metformin.\n### Response:\n"
inputs = tokenizer(test_prompt, return_tensors="pt").to("mps")

dpo_model.generation_config.max_length = None
outputs = dpo_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True
)

print("\nDPO Aligned Model Response:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))